# Notebook 2: detectors under label drift (ClawHub)

**Runtime:** Runtime → Change runtime type → **G4 GPU** (or H100/A100), **High-RAM on** → Save.

1. Run the first cell. It asks you to upload **`labels.parquet`** (8.7 MB) from
   `~/Documents/research/project-01_clawhub-temporal-drift/02_data/labels.parquet`.
2. Then **Run all** (Runtime → Run after). Total about 2–3 hours:
   fetch texts from Hugging Face (~30–45 min) → TF-IDF baseline (~20 min) → fine-tuned ModernBERT (~1.5 h).
3. At the end your browser downloads **`clawhub_detectors_results.zip`**. Put it in `~/Downloads`.

In [ ]:
from google.colab import files
import os, shutil
up = files.upload()  # choose labels.parquet
name = next(iter(up))
shutil.move(name, "/content/labels.parquet")
print("uploaded", name, os.path.getsize("/content/labels.parquet"), "bytes")

In [ ]:
!pip -q install "transformers==5.19.0"
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
%%writefile build_snapshots.py
"""build_snapshots.py - rebuild every historical snapshot of OpenClaw/clawhub-security-signals-live.

For each day the dataset was published (one Hugging Face commit per day), stream data/latest.jsonl
at that exact commit and keep:
  panel.parquet   one row per (snapshot, skill version): verdicts and scanner signals, content hashes
  texts.parquet   deduplicated texts keyed by sha256 (SKILL.md, bundle, ClawScan summary, ClawScan context)
  snapshots.json  per snapshot: commit, date, manifest, row counts, parse problems

Raw files are deleted after parsing, so disk use stays at one snapshot (~3 GB) at a time.
Everything is pinned to commit SHAs, so the build is reproducible.
"""
from __future__ import annotations

import hashlib
import json
import os
import sys
import time

REPO = "OpenClaw/clawhub-security-signals-live"
BUNDLE_MAX_CHARS = 60_000  # per skill version, for the text model; full size is recorded separately

PANEL_COLS = [
    "snapshot", "commit", "id", "skill_slug", "skill_version",
    "md_sha", "md_chars", "bundle_sha", "bundle_files", "bundle_bytes",
    "clawscan_verdict", "clawscan_confidence", "clawscan_model", "summary_sha", "context_sha",
    "static_status", "static_finding_count", "static_reason_codes",
    "virustotal_status", "virustotal_malicious_count", "virustotal_suspicious_count",
    "virustotal_harmless_count", "virustotal_undetected_count",
    "skillspector_status", "skillspector_score", "skillspector_severity", "skillspector_issue_count",
    "skillspector_issue_codes", "skillspector_issue_categories",
]


def log(*a):
    print(time.strftime("[%H:%M:%S]"), *a, flush=True)


def sha(s: str) -> str:
    return hashlib.sha256(s.encode("utf-8", "surrogatepass")).hexdigest()


def _list(v):
    if v is None:
        return []
    if isinstance(v, str):
        try:
            v = json.loads(v)
        except Exception:
            return [v]
    return list(v) if isinstance(v, (list, tuple)) else [v]


def _int(v):
    try:
        return None if v is None or v == "" else int(v)
    except (TypeError, ValueError):
        return None


def bundle_text(files):
    """Concatenate bundled files as '### path' blocks (sorted by path), plus a content signature."""
    files = sorted((f for f in files if isinstance(f, dict)), key=lambda f: str(f.get("path", "")))
    parts, sig, total = [], [], 0
    for f in files:
        path, content = str(f.get("path", "")), f.get("content")
        content = "" if content is None else str(content)
        sig.append(f"{path}:{f.get('sha256') or sha(content)}")
        total += _int(f.get("sizeBytes")) or len(content)
        parts.append(f"### {path}\n{content}")
    return "\n\n".join(parts), sha("\n".join(sig)), len(files), total


def parse_row(obj, snapshot, commit):
    """One JSONL object -> (panel row, {sha: (kind, text)})."""
    md = obj.get("skill_md_content") or ""
    btext, bsig, nfiles, nbytes = bundle_text(_list(obj.get("skill_bundle_content")))
    summary = obj.get("clawscan_summary") or ""
    ctx = obj.get("clawscan_context")
    ctx = ctx if isinstance(ctx, str) else json.dumps(ctx, sort_keys=True, ensure_ascii=False)
    md_sha, sum_sha, ctx_sha = sha(md), sha(summary), sha(ctx or "")
    row = {
        "snapshot": snapshot, "commit": commit,
        "id": obj.get("id"), "skill_slug": obj.get("skill_slug"), "skill_version": obj.get("skill_version"),
        "md_sha": md_sha, "md_chars": len(md), "bundle_sha": bsig, "bundle_files": nfiles, "bundle_bytes": nbytes,
        "clawscan_verdict": obj.get("clawscan_verdict"), "clawscan_confidence": obj.get("clawscan_confidence"),
        "clawscan_model": obj.get("clawscan_model"), "summary_sha": sum_sha, "context_sha": ctx_sha,
        "static_status": obj.get("static_status"), "static_finding_count": _int(obj.get("static_finding_count")),
        "static_reason_codes": "|".join(map(str, _list(obj.get("static_reason_codes")))),
        "virustotal_status": obj.get("virustotal_status"),
        "virustotal_malicious_count": _int(obj.get("virustotal_malicious_count")),
        "virustotal_suspicious_count": _int(obj.get("virustotal_suspicious_count")),
        "virustotal_harmless_count": _int(obj.get("virustotal_harmless_count")),
        "virustotal_undetected_count": _int(obj.get("virustotal_undetected_count")),
        "skillspector_status": obj.get("skillspector_status"),
        "skillspector_score": _int(obj.get("skillspector_score")),
        "skillspector_severity": obj.get("skillspector_severity"),
        "skillspector_issue_count": _int(obj.get("skillspector_issue_count")),
        "skillspector_issue_codes": "|".join(map(str, _list(obj.get("skillspector_issue_codes")))),
        "skillspector_issue_categories": "|".join(map(str, _list(obj.get("skillspector_issue_categories")))),
    }
    texts = {md_sha: ("md", md), bsig: ("bundle", btext[:BUNDLE_MAX_CHARS]),
             sum_sha: ("summary", summary), ctx_sha: ("context", ctx or "")}
    return row, texts


def parse_jsonl(path, snapshot, commit, seen_texts, text_sink):
    """Stream a JSONL file. New texts go to text_sink (list); returns (rows, stats)."""
    rows, bad, n = [], 0, 0
    with open(path, "rb") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            n += 1
            try:
                obj = json.loads(line)
            except Exception:
                bad += 1
                continue
            row, texts = parse_row(obj, snapshot, commit)
            rows.append(row)
            for h, (kind, t) in texts.items():
                if h not in seen_texts:
                    seen_texts.add(h)
                    text_sink.append({"sha": h, "kind": kind, "text": t})
    return rows, {"lines": n, "bad_lines": bad, "rows": len(rows)}


def daily_commits():
    """Last commit of each day, oldest first: [(YYYY-MM-DD, sha, title)]."""
    from huggingface_hub import HfApi
    commits = HfApi().list_repo_commits(REPO, repo_type="dataset")
    by_day = {}
    for c in sorted(commits, key=lambda c: c.created_at):
        by_day[c.created_at.strftime("%Y-%m-%d")] = (c.commit_id, c.title)
    return [(d, s, t) for d, (s, t) in sorted(by_day.items())]


def build(out_dir, work_dir, limit=None):
    import pandas as pd
    from huggingface_hub import hf_hub_download
    os.makedirs(out_dir, exist_ok=True)
    os.makedirs(work_dir, exist_ok=True)
    days = daily_commits()
    if limit:
        days = days[:limit]
    log(f"{len(days)} daily snapshots: {days[0][0]} -> {days[-1][0]}")
    seen, all_rows, text_parts, meta = set(), [], [], []
    for i, (day, commit, title) in enumerate(days, 1):
        t0 = time.time()
        try:
            data = hf_hub_download(REPO, "data/latest.jsonl", repo_type="dataset", revision=commit,
                                   cache_dir=work_dir)
        except Exception as e:  # a commit without the data file (e.g. the initial commit)
            log(f"[{i}/{len(days)}] {day} {commit[:8]}: no data file ({type(e).__name__}), skipped")
            meta.append({"snapshot": day, "commit": commit, "title": title, "skipped": str(e)[:200]})
            continue
        try:
            mpath = hf_hub_download(REPO, "metadata/latest-manifest.json", repo_type="dataset", revision=commit,
                                    cache_dir=work_dir)
            manifest = json.load(open(mpath))
        except Exception as e:
            manifest = {"error": str(e)[:200]}
        sink = []
        rows, stats = parse_jsonl(data, day, commit, seen, sink)
        all_rows.extend(rows)
        if sink:
            part = os.path.join(work_dir, f"texts_{i:03d}.parquet")
            pd.DataFrame(sink).to_parquet(part, compression="zstd", index=False)
            text_parts.append(part)
        meta.append({"snapshot": day, "commit": commit, "title": title, "manifest": manifest,
                     "file_bytes": os.path.getsize(data), "new_texts": len(sink), **stats})
        # free disk: remove downloaded blobs for this revision
        for root, _, files in os.walk(work_dir):
            for fn in files:
                p = os.path.join(root, fn)
                if not fn.startswith("texts_") and os.path.getsize(p) > 50_000_000:
                    os.remove(p)
        log(f"[{i}/{len(days)}] {day} {commit[:8]}: {stats['rows']} rows, {stats['bad_lines']} bad, "
            f"{len(sink)} new texts, {time.time() - t0:.0f}s")
    panel = pd.DataFrame(all_rows, columns=PANEL_COLS)
    panel.to_parquet(os.path.join(out_dir, "panel.parquet"), compression="zstd", index=False)
    texts = pd.concat([pd.read_parquet(p) for p in text_parts], ignore_index=True) if text_parts else pd.DataFrame()
    texts.to_parquet(os.path.join(out_dir, "texts.parquet"), compression="zstd", index=False)
    json.dump(meta, open(os.path.join(out_dir, "snapshots.json"), "w"), indent=1, default=str)
    log(f"done: panel {panel.shape}, texts {texts.shape}")
    return panel, texts, meta


if __name__ == "__main__":
    # local test on an already-downloaded JSONL:  python build_snapshots.py --test file.jsonl
    if len(sys.argv) > 2 and sys.argv[1] == "--test":
        seen, sink = set(), []
        rows, stats = parse_jsonl(sys.argv[2], "test", "local", seen, sink)
        print(stats, "texts:", len(sink))
        import collections
        print(collections.Counter(r["clawscan_verdict"] for r in rows))
        print(rows[0])
    else:
        build(sys.argv[1], sys.argv[2], int(sys.argv[3]) if len(sys.argv) > 3 else None)

In [ ]:
%%writefile colab_texts.py
"""colab_texts.py - fetch the skill texts needed by labels.parquet straight from Hugging Face (run on Colab).

Walks the daily snapshots newest -> oldest, hashes each row exactly like build_snapshots.py, and keeps the
(truncated) SKILL.md and bundle text of every md_sha / bundle_sha listed in labels.parquet. Stops early once
everything is found. Returns labels joined with md_text / bundle_text.
"""
import json
import os
import time

import pandas as pd

import build_snapshots as bs


def fetch_texts(labels: pd.DataFrame, work_dir: str, md_max=12_000, bundle_max=8_000, days=None,
                download=None):
    need_md, need_b = set(labels.md_sha.dropna()), set(labels.bundle_sha.dropna())
    md_txt, b_txt = {}, {}
    if days is None:
        days = bs.daily_commits()
    if download is None:
        from huggingface_hub import hf_hub_download

        def download(commit):
            return hf_hub_download(bs.REPO, "data/latest.jsonl", repo_type="dataset", revision=commit,
                                   cache_dir=work_dir)
    for day, commit, _ in sorted(days, reverse=True):
        if not (need_md - md_txt.keys()) and not (need_b - b_txt.keys()):
            break
        t0 = time.time()
        try:
            path = download(commit)
        except Exception as e:
            bs.log(f"{day}: no data ({type(e).__name__})")
            continue
        with open(path, "rb") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                obj = json.loads(line)
                md = obj.get("skill_md_content") or ""
                h = bs.sha(md)
                if h in need_md and h not in md_txt:
                    md_txt[h] = md[:md_max]
                btext, bsig, _, _ = bs.bundle_text(bs._list(obj.get("skill_bundle_content")))
                if bsig in need_b and bsig not in b_txt:
                    b_txt[bsig] = btext[:bundle_max]
        if os.path.getsize(path) > 50_000_000 and os.path.realpath(path) != path:
            os.remove(os.path.realpath(path))  # free disk (HF cache blob)
        bs.log(f"{day}: md {len(md_txt)}/{len(need_md)}, bundle {len(b_txt)}/{len(need_b)} ({time.time() - t0:.0f}s)")
    out = labels.copy()
    out["md_text"] = out.md_sha.map(md_txt).fillna("")
    out["bundle_text"] = out.bundle_sha.map(b_txt).fillna("")
    out["text_found"] = out.md_sha.isin(md_txt.keys())
    bs.log(f"texts found for {out.text_found.mean():.1%} of skill versions")
    return out

In [ ]:
%%writefile detect.py
"""detect.py - detectors trained on scanner labels, evaluated against pre- and post-rescan labels.

Experiments (one per label source):
  ss  SkillSpector (rescanned 14-21 Sep 2026)   <- the label-drift experiment
  vt  VirusTotal                                <- control
  st  static analysis                           <- control
For each: train on *_old labels (pre-rescan) of the train split; select threshold on valid (*_old);
score the test split and report metrics against *_old AND *_new labels for the same test rows.
Also trains on *_new (post-rescan) to show what an updated detector would achieve.
Secondary: registry verdict (verdict_any), rare positives -> PR-AUC.

Models: 'tfidf' (char+word TF-IDF + logistic regression, CPU) and 'bert' (fine-tuned transformer, GPU).
"""
from __future__ import annotations

import json
import os
import time

import numpy as np
import pandas as pd
from sklearn.metrics import average_precision_score, f1_score, precision_score, recall_score, roc_auc_score


def log(*a):
    print(time.strftime("[%H:%M:%S]"), *a, flush=True)


def load(path):
    df = pd.read_parquet(path)
    df["text"] = df["md_text"].fillna("") + "\n\n#### BUNDLE\n" + df["bundle_text"].fillna("")
    return df


def metrics(y, s, thr):
    y = np.asarray(y).astype(int)
    p = (np.asarray(s) >= thr).astype(int)
    out = {"n": int(len(y)), "pos": int(y.sum()), "pos_rate": float(y.mean()) if len(y) else float("nan")}
    if len(np.unique(y)) == 2:
        out.update(roc_auc=float(roc_auc_score(y, s)), pr_auc=float(average_precision_score(y, s)))
    out.update(f1=float(f1_score(y, p, zero_division=0)), precision=float(precision_score(y, p, zero_division=0)),
               recall=float(recall_score(y, p, zero_division=0)), accuracy=float((p == y).mean()))
    return out


def best_threshold(y, s):
    """Threshold maximising F1 on the validation set."""
    y = np.asarray(y).astype(int)
    qs = np.unique(np.quantile(s, np.linspace(0.01, 0.99, 197)))
    f1s = [f1_score(y, (s >= t).astype(int), zero_division=0) for t in qs]
    return float(qs[int(np.argmax(f1s))])


def bootstrap_ci(y, s, fn, n=1000, seed=0):
    rng = np.random.default_rng(seed)
    y, s = np.asarray(y), np.asarray(s)
    vals = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if len(np.unique(y[i])) == 2:
            vals.append(fn(y[i], s[i]))
    return [float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))] if vals else [float("nan")] * 2


# ----------------------------------------------------------------------------- models

class TfidfModel:
    name = "tfidf"

    def fit(self, tr_text, tr_y, seed=0):
        from sklearn.feature_extraction.text import TfidfVectorizer
        from sklearn.linear_model import LogisticRegression
        self.vw = TfidfVectorizer(max_features=300_000, ngram_range=(1, 2), min_df=3, sublinear_tf=True,
                                  token_pattern=r"(?u)\b\w[\w\-\.]+\b", dtype=np.float32)
        X = self.vw.fit_transform(tr_text)
        self.clf = LogisticRegression(C=4.0, max_iter=2000, class_weight="balanced", random_state=seed)
        self.clf.fit(X, tr_y)
        return self

    def score(self, text):
        X = self.vw.transform(text)
        return self.clf.predict_proba(X)[:, 1]


class BertModel:
    name = "bert"

    def __init__(self, model_name="answerdotai/ModernBERT-base", max_len=1024, epochs=2, batch=16, lr=3e-5,
                 tiny=False):
        self.model_name, self.max_len, self.epochs, self.batch, self.lr = model_name, max_len, epochs, batch, lr
        self.tiny = tiny  # local smoke test: random tiny network, same tokenizer and code path

    def fit(self, tr_text, tr_y, seed=0):
        import torch
        from transformers import AutoModelForSequenceClassification, AutoTokenizer
        torch.manual_seed(seed)
        np.random.seed(seed)
        self.dev = "cuda" if torch.cuda.is_available() else "cpu"
        self.tok = AutoTokenizer.from_pretrained(self.model_name)
        if self.tiny:
            from transformers import AutoConfig
            cfg = AutoConfig.from_pretrained(self.model_name, num_labels=2, hidden_size=32, intermediate_size=64,
                                             num_hidden_layers=2, num_attention_heads=2)
            self.model = AutoModelForSequenceClassification.from_config(cfg).to(self.dev)
        else:
            self.model = AutoModelForSequenceClassification.from_pretrained(self.model_name, num_labels=2).to(self.dev)
        y = np.asarray(tr_y).astype(int)
        w = torch.tensor([1.0, float(max(1.0, (1 - y.mean()) / max(y.mean(), 1e-6)) ** 0.5)], dtype=torch.float32, device=self.dev)
        opt = torch.optim.AdamW(self.model.parameters(), lr=self.lr, weight_decay=0.01)
        steps = self.epochs * int(np.ceil(len(y) / self.batch))
        sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=self.lr, total_steps=steps, pct_start=0.1)
        amp = self.dev == "cuda"
        dt = torch.bfloat16 if (amp and torch.cuda.is_bf16_supported()) else torch.float16
        texts = list(tr_text)
        self.model.train()
        step = 0
        for ep in range(self.epochs):
            order = np.random.permutation(len(y))
            t0, losses = time.time(), []
            for b in range(0, len(order), self.batch):
                idx = order[b:b + self.batch]
                enc = self.tok([texts[i] for i in idx], truncation=True, max_length=self.max_len, padding=True,
                               return_tensors="pt").to(self.dev)
                with torch.autocast(device_type="cuda", dtype=dt, enabled=amp):
                    logits = self.model(**enc).logits.float()
                loss = torch.nn.functional.cross_entropy(logits, torch.tensor(y[idx], device=self.dev), weight=w)
                opt.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                opt.step()
                sched.step()
                step += 1
                losses.append(float(loss))
                if step % 200 == 0:
                    log(f"  epoch {ep + 1} step {step}/{steps} loss {np.mean(losses[-200:]):.4f}")
            log(f"  epoch {ep + 1} done, mean loss {np.mean(losses):.4f}, {time.time() - t0:.0f}s")
        return self

    def score(self, text):
        import torch
        self.model.eval()
        out, texts = [], list(text)
        amp = self.dev == "cuda"
        dt = torch.bfloat16 if (amp and torch.cuda.is_bf16_supported()) else torch.float16
        with torch.inference_mode():
            for b in range(0, len(texts), self.batch * 2):
                enc = self.tok(texts[b:b + self.batch * 2], truncation=True, max_length=self.max_len, padding=True,
                               return_tensors="pt").to(self.dev)
                with torch.autocast(device_type="cuda", dtype=dt, enabled=amp):
                    out.append(torch.softmax(self.model(**enc).logits.float(), -1)[:, 1].cpu().numpy())
        return np.concatenate(out) if out else np.zeros(0)


# ----------------------------------------------------------------------------- experiments

def run_label_drift(df, make_model, source, seed=0, max_train=None):
    """Train on {source}_old (and separately {source}_new); test against both label versions."""
    res = {"source": source, "seed": seed}
    both = df[df[f"{source}_old"].notna() & df[f"{source}_new"].notna()]
    tr, va, te = (both[both.split == s] for s in ("train", "valid", "test"))
    if max_train and len(tr) > max_train:
        tr = tr.sample(max_train, random_state=seed)
    res["sizes"] = {"train": len(tr), "valid": len(va), "test": len(te)}
    res["test_label_changed"] = int((te[f"{source}_old"] != te[f"{source}_new"]).sum())
    for train_on in ("old", "new"):
        col = f"{source}_{train_on}"
        t0 = time.time()
        m = make_model().fit(tr.text, tr[col].astype(int), seed=seed)
        s_va, s_te = m.score(va.text), m.score(te.text)
        thr = best_threshold(va[col].astype(int), s_va)
        r = {"threshold": thr, "minutes": round((time.time() - t0) / 60, 2)}
        for eval_on in ("old", "new"):
            y = te[f"{source}_{eval_on}"].astype(int).values
            r[f"test_vs_{eval_on}"] = metrics(y, s_te, thr)
            r[f"test_vs_{eval_on}"]["roc_auc_ci"] = bootstrap_ci(y, s_te, roc_auc_score, n=300, seed=seed)
        res[f"trained_on_{train_on}"] = r
        res.setdefault("scores", {})[f"trained_on_{train_on}"] = s_te.astype(float).tolist()
        log(f"{m.name}/{source}/train={train_on}: "
            f"AUC vs old {r['test_vs_old'].get('roc_auc', float('nan')):.3f}, "
            f"vs new {r['test_vs_new'].get('roc_auc', float('nan')):.3f} | "
            f"F1 vs old {r['test_vs_old']['f1']:.3f}, vs new {r['test_vs_new']['f1']:.3f}")
    res["test_keys"] = te.k.tolist()
    return res


def run_verdict(df, make_model, seed=0):
    d = df.copy()
    tr, va, te = (d[d.split == s] for s in ("train", "valid", "test"))
    y_tr = tr.verdict_any.astype(int)
    m = make_model().fit(tr.text, y_tr, seed=seed)
    s_va, s_te = m.score(va.text), m.score(te.text)
    thr = best_threshold(va.verdict_any.astype(int), s_va) if va.verdict_any.sum() else 0.5
    r = {"source": "verdict", "seed": seed, "sizes": {"train": len(tr), "valid": len(va), "test": len(te)},
         "test": metrics(te.verdict_any.astype(int), s_te, thr), "threshold": thr}
    log(f"{m.name}/verdict: test pos {r['test']['pos']}, PR-AUC {r['test'].get('pr_auc', float('nan')):.3f}, "
        f"AUC {r['test'].get('roc_auc', float('nan')):.3f}")
    return r


def run_all(data_path, out_dir, model="tfidf", seeds=(0,), sources=("ss", "vt", "st"), max_train=None,
            bert_kwargs=None, verdict=True):
    os.makedirs(out_dir, exist_ok=True)
    df = load(data_path)
    log(f"data: {len(df)} skill versions; splits {df.split.value_counts().to_dict()}")
    make = (lambda: TfidfModel()) if model == "tfidf" else (lambda: BertModel(**(bert_kwargs or {})))
    for seed in seeds:
        for src in sources:
            path = os.path.join(out_dir, f"{model}_{src}_s{seed}.json")
            if os.path.exists(path):
                log(f"skip {path} (done)")
                continue
            r = run_label_drift(df, make, src, seed=seed, max_train=max_train)
            json.dump(r, open(path, "w"))
        if verdict:
            path = os.path.join(out_dir, f"{model}_verdict_s{seed}.json")
            if not os.path.exists(path):
                json.dump(run_verdict(df, make, seed=seed), open(path, "w"))
    log("all done")


if __name__ == "__main__":
    import sys
    run_all(sys.argv[1], sys.argv[2], model=sys.argv[3] if len(sys.argv) > 3 else "tfidf",
            max_train=int(sys.argv[4]) if len(sys.argv) > 4 else None)

## 1. Fetch skill texts (needed hashes only)

In [ ]:
import pandas as pd, colab_texts as ct
labels = pd.read_parquet("/content/labels.parquet")
DATA = "/content/detection_set.parquet"
if not os.path.exists(DATA):
    det = ct.fetch_texts(labels, "/content/hf_work")
    det.to_parquet(DATA, compression="zstd")
det = pd.read_parquet(DATA)
print(det.groupby("split").text_found.mean())

## 2. TF-IDF baseline (CPU part, about 20 min)

In [ ]:
import detect
OUT = "/content/results"
detect.run_all(DATA, OUT, model="tfidf", sources=("ss", "st", "vt"))

## 3. Fine-tuned ModernBERT (GPU, about 1.5 h)

In [ ]:
detect.run_all(DATA, OUT, model="bert", sources=("ss", "st"), verdict=True,
               bert_kwargs=dict(model_name="answerdotai/ModernBERT-base", max_len=1024, epochs=2, batch=16, lr=3e-5))

## 4. Download results

In [ ]:
import shutil
z = shutil.make_archive("/content/clawhub_detectors_results", "zip", OUT)
print(z, os.path.getsize(z) // 1_000_000, "MB")
files.download(z)